In [0]:
# Auto Loader: Read CSV from input path and write to Bronze Delta table
from pyspark.sql.functions import *

input_path = "/Volumes/data_modeliing_demo/source/demo_volume/scd2/input/"
checkpoint_path = "/Volumes/data_modeliing_demo/source/demo_volume/scd2/checkpoint/"
target_table = "data_modeliing_demo.bronze.scd2"

print(f"Input path    : {input_path}")
print(f"Checkpoint    : {checkpoint_path}")
print(f"Target table  : {target_table}")
print("-" * 80)

In [0]:
# Build the Auto Loader read stream
autoloader_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(input_path)
)

In [0]:
# Write stream to the Delta table with trigger(availableNow=True) for batch-mode processing
query = (autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

print(f"\n✅ Data written to {target_table} successfully.")

In [0]:
%sql
select * from data_modeliing_demo.bronze.scd2 order by customer_id,last_updated_ts;